In [ ]:
import json
import re
import numpy as np
from nltk.corpus import reuters

print("Loading vocabulary...")
with open("data/word2id.json", "r") as f:
    word2id = json.load(f)
print(f"Vocabulary size: {len(word2id)}")

print("\nLoading embeddings (using Skip-gram NEG)...")
E = np.load("embeddings/sg_neg_embeddings.npy")
print(f"Embeddings shape: {E.shape}")

In [ ]:
def normalize_rows(M, eps=1e-9):
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    return M / (norms + eps)

E_n = normalize_rows(E)

def text_to_ids(text):
    tokens = re.findall(r"[a-zA-Z]+", text.lower())
    return [word2id.get(t, 0) for t in tokens]

def paragraph_vector(text):
    ids = text_to_ids(text)
    ids = [i for i in ids if i != 0]  # Drop UNK
    if len(ids) == 0:
        return None
    return E_n[ids].mean(axis=0)

print("Helper functions defined")

In [ ]:
MAX_DOCS = 2000
MIN_CHARS = 80

print(f"Loading Reuters corpus (first {MAX_DOCS} documents)...")
fileids = reuters.fileids()[:MAX_DOCS]

paragraphs = []
for fid in fileids:
    raw = reuters.raw(fid)
    # Split on blank lines
    for p in re.split(r"\n\s*\n", raw):
        p = p.strip()
        if len(p) >= MIN_CHARS:
            paragraphs.append(p)

print(f"Found {len(paragraphs)} paragraphs (>= {MIN_CHARS} chars)")

In [ ]:
print("Computing paragraph vectors...")
vecs = []
texts = []

for i, p in enumerate(paragraphs):
    if (i + 1) % 500 == 0:
        print(f"Processed {i+1}/{len(paragraphs)} paragraphs...")
    
    v = paragraph_vector(p)
    if v is None:
        continue
    
    vecs.append(v.astype(np.float32))
    texts.append(p)

print(f"\nKept {len(texts)} paragraphs with valid embeddings")

# Stack and normalize
V = np.vstack(vecs)
V_n = normalize_rows(V)

print(f"Paragraph vectors shape: {V_n.shape}")

In [ ]:
print("Saving data for web application...")

# Save paragraph vectors
np.save("app/paragraph_vectors.npy", V_n)
print("✓ Saved: app/paragraph_vectors.npy")

# Save paragraph texts
with open("app/paragraph_texts.json", "w") as f:
    json.dump(texts, f)
print("✓ Saved: app/paragraph_texts.json")

print("\nExample paragraph:")
print("-" * 80)
print(texts[0][:200] + "...")
print("-" * 80)
print("\nWeb app data preparation complete!")